# ALIGN-PARETO — M3 pre-check: the λ dial
λ blends the weights of our M2 SFT model (λ = 0, unaligned) and Qwen's `Qwen/Qwen3-0.6B` (λ = 1, aligned):
W(λ) = W_sft + λ · (W_qwen3 − W_sft). Before the full M3 sweep this checks that λ = 0 and λ = 1 reproduce the two
models exactly, and that every λ gives replies that stop and contain no stray characters (16 prompts, 4 per quadrant).

**Kaggle setup:** inputs `align-pareto-src` and `align-pareto-m2-adapters` attached, **GPU T4**, internet on.
Run from this machine with `kaggle/run_on_kaggle.sh m3_interp_check` (publish the adapter first with
`kaggle/publish_adapters.sh`).

In [ ]:
# 1. Environment (one GPU: the trainers would otherwise wrap the model in DataParallel)
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
import sys, json, platform, shutil, glob, subprocess
from pathlib import Path
import importlib.metadata as md

print("Python", platform.python_version())
import torch
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), f"| VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
else:
    print("No GPU: M2 needs one (Settings -> Accelerator -> GPU T4)")

In [ ]:
# 2. Dependencies (pinned to the versions the code was tested with)
PINNED = ["transformers==5.22.2", "trl==1.17.1", "peft==0.21.2", "datasets"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINNED], check=True)
# Kaggle preinstalls torchao 0.10; peft 0.21 refuses to wrap layers while an old torchao is importable (unused here)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
for pkg in ["torch", "transformers", "trl", "peft", "accelerate", "datasets", "huggingface_hub"]:
    try:
        print(f"{pkg:16s} {md.version(pkg)}")
    except md.PackageNotFoundError:
        print(f"{pkg:16s} MISSING")

In [ ]:
# 3. Project code: Kaggle dataset `align-pareto-src` -> /kaggle/working/align-pareto; locally, the repo itself
ON_KAGGLE = Path("/kaggle/working").exists()
if ON_KAGGLE:
    hits = glob.glob("/kaggle/input/**/src/alignment/train_sft.py", recursive=True)
    assert hits, "align-pareto-src is missing or out of date (no src/alignment/train_sft.py): attach its latest version."
    ROOT = Path("/kaggle/working/align-pareto")
    shutil.copytree(Path(hits[0]).parents[2], ROOT, dirs_exist_ok=True)
else:
    ROOT = Path.cwd().parent
assert (ROOT / "data/eval/eval_prompts.csv").exists(), "data/eval/eval_prompts.csv missing: attach the latest align-pareto-src"
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)

In [ ]:
# 4. SFT adapter from the Kaggle dataset align-pareto-m2-adapters -> models/m2/sft_adapter
if ON_KAGGLE:
    hits = glob.glob("/kaggle/input/**/models/m2/sft_adapter/adapter_config.json", recursive=True)
    assert hits, "align-pareto-m2-adapters is missing: run kaggle/publish_adapters.sh and attach it."
    shutil.copytree(Path(hits[0]).parent, ROOT / "models/m2/sft_adapter", dirs_exist_ok=True)
assert (ROOT / "models/m2/sft_adapter/adapter_config.json").exists(), "models/m2/sft_adapter missing"

In [ ]:
# 5. Run the check
from src.check_interpolation import check_interpolation
metrics = check_interpolation(ROOT)
RES = ROOT / "results/m3_interp_check"
if ON_KAGGLE:
    shutil.rmtree(ROOT / "models", ignore_errors=True)   # the adapter is an input, not an output
print(json.dumps(metrics["endpoint_check"], indent=2))

In [ ]:
# 6. Per λ: share of replies that stop (want ~0.6+ like M2), stray characters (want 0), rough refusal shares
import pandas as pd
pd.set_option("display.max_colwidth", 300)
display(pd.DataFrame(metrics["per_lambda"]).T)
replies = pd.read_csv(RES / "replies.csv")
display(replies[["quadrant", "prompt"] + [c for c in replies if c.startswith("reply_lambda_")]])